In [1]:
#!/usr/bin/env python3
"""
bayes_missing.py -- joint Bayesian logistic regression with missing covariates
and per-variable selection models for (possibly non-ignorable) missingness.

Pure numpy/scipy Gibbs sampler (no PyMC/Stan needed).

Model
-----
Base variables z_i (q, never missing), vitals x_i (p, partly missing), outcome y_i.

 1. Vitals  : x_i | z_i ~ MVN(B' [1, z_i], Sigma)              B ~ matrix-normal, Sigma ~ IW
 2. Outcome : y_i | z_i, x_i ~ Bernoulli(logit^-1(a + z_i b_z + x_i b_x))
              (Polya-gamma augmentation -> all conditionals conjugate)
 3. Missingness (only for vitals you list in `selection`; others are treated as ignorable)
              R_ij = 1[observed],  P(R_ij = 1) = Phi(g0 + z_i g_z + y_i g_y + g_x * x_ij)
              (Albert-Chib probit augmentation)

    * g_x is the NON-IGNORABLE part: missingness depends on the vital's own value.
      selection[name] = None   -> g_x free, prior N(0, tau_gx^2)
      selection[name] = delta  -> g_x FIXED at delta (pattern-mixture-style sensitivity analysis)
    * vitals not in `selection` are assumed MCAR/MAR given (z, y, other observed values):
      the likelihood factorises and no missingness model is needed.

Missing x_ij are latent parameters: each Gibbs sweep draws them from their full
conditional, which combines (i) the MVN vitals model, (ii) the outcome likelihood and
(iii) the selection model.  So imputation uncertainty propagates into every credible
interval, and the outcome informs the imputations (congenial imputation).
"""
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from scipy import linalg
from scipy.special import log_ndtr, ndtri, ndtri_exp
from scipy.stats import invwishart


# --------------------------------------------------------------------------- samplers
def sample_pg1(c, rng, K=200):
    """Draws from Polya-gamma PG(1, c): truncated series + mean-corrected remainder."""
    c = np.abs(c)
    k = np.arange(1, K + 1) - 0.5
    denom = k[None, :] ** 2 + (c[:, None] ** 2) / (4 * np.pi ** 2)
    g = rng.standard_gamma(1.0, size=denom.shape)
    s = (g / denom).sum(1) / (2 * np.pi ** 2)
    full_mean = np.where(c > 1e-6, np.tanh(c / 2) / (2 * np.maximum(c, 1e-6)), 0.25)
    trunc_mean = (1.0 / denom).sum(1) / (2 * np.pi ** 2)
    return s + (full_mean - trunc_mean)


def draw_latent_probit(mean, observed, rng):
    """w ~ N(mean, 1) truncated to w>0 where observed else w<0 (numerically stable)."""
    u = np.clip(rng.random(mean.shape), 1e-12, 1.0)
    # R=0: w = m + Phi^-1(u Phi(-m)) ; R=1: w = m - Phi^-1(u Phi(m))
    z0 = ndtri_exp(np.log(u) + log_ndtr(-mean))
    z1 = ndtri_exp(np.log(u) + log_ndtr(mean))
    return np.where(observed, mean - z1, mean + z0)


def _chol_sample(prec, rhs, rng):
    """Draw from N(prec^-1 rhs, prec^-1)."""
    L = np.linalg.cholesky(prec)
    mean = linalg.cho_solve((L, True), rhs)
    eps = rng.standard_normal(len(rhs))
    return mean + linalg.solve_triangular(L, eps, lower=True, trans="T")


# --------------------------------------------------------------------------- data prep
def prepare(df, base_cols, vital_cols, outcome_col, log_vars=()):
    """Standardise base variables (fully observed) and vitals (on observed values).

    Returns a dict; coefficients come back per 1 'model unit' and are mapped to
    original units with `x_scale`.
    """
    base = df[base_cols].to_numpy(float)
    z_loc, z_scale = base.mean(0), base.std(0)
    Z = (base - z_loc) / z_scale
    X = df[vital_cols].to_numpy(float).copy()
    for j, name in enumerate(vital_cols):
        if name in log_vars:
            X[:, j] = np.log(X[:, j])
    x_loc, x_scale = np.nanmean(X, 0), np.nanstd(X, 0)
    X = (X - x_loc) / x_scale
    return dict(Z=Z, X=X, y=df[outcome_col].to_numpy(float), z_scale=z_scale,
                x_loc=x_loc, x_scale=x_scale, base_cols=list(base_cols), vital_cols=list(vital_cols))


# --------------------------------------------------------------------------- one chain
def run_chain(Z, X, y, selection, names, n_iter=3000, n_burn=1000, seed=0,
              tau_beta=2.5, tau_g=2.0, tau_gx=1.0, tau_B=np.sqrt(10.0),
              prior_obs_rate=None, tau_g0_informative=0.5, verbose=False):
    """Gibbs sampler for one chain.  `selection`: {vital_name: None | fixed g_x}."""
    rng = np.random.default_rng(seed)
    N, p = X.shape
    q = Z.shape[1]
    obs = ~np.isnan(X)
    miss = ~obs
    Xc = np.where(obs, X, 0.0)                    # start imputations at the (standardised) mean
    kb = 1 + q + p
    kappa = y - 0.5

    # --- vitals model
    Zc = np.column_stack([np.ones(N), Z])
    K0 = np.eye(q + 1) / tau_B ** 2
    V_B = np.linalg.inv(Zc.T @ Zc + K0)
    L_VB = np.linalg.cholesky(V_B)
    B = V_B @ Zc.T @ Xc
    Sigma = np.cov((Xc - Zc @ B).T) + 1e-3 * np.eye(p)
    nu0, Psi0 = p + 2, np.eye(p)

    # --- outcome model
    beta = np.zeros(kb)
    P0 = np.eye(kb) / tau_beta ** 2

    # --- selection models
    sel = [j for j, nm in enumerate(names) if nm in selection]
    M = np.column_stack([np.ones(N), Z, y])        # [1, z, y]
    km = M.shape[1]
    fixed = {j: selection[names[j]] for j in sel}
    gam = {j: np.zeros(km + 1) for j in sel}
    for j in sel:
        if fixed[j] is not None:
            gam[j][km] = fixed[j]
    w = np.zeros((N, p))
    prior_m = {}
    prior_sd = {}
    for j in sel:
        m0 = np.zeros(km + 1)
        sd = np.full(km + 1, tau_g)
        sd[km] = tau_gx
        if prior_obs_rate is not None and names[j] in prior_obs_rate:       # informative prior on the
            m0[0] = ndtri(np.clip(prior_obs_rate[names[j]], 1e-3, 1 - 1e-3))  # observation rate
            sd[0] = tau_g0_informative
        prior_m[j], prior_sd[j] = m0, sd
        w[:, j] = np.where(obs[:, j], 0.5, -0.5)

    n_keep = n_iter - n_burn
    out = dict(
        beta=np.zeros((n_keep, kb)),
        gamma=np.zeros((n_keep, len(sel), km + 1)),
        imp_mean=np.zeros((n_keep, p)),
        Sigma=np.zeros((n_keep, p, p)),
        B=np.zeros((n_keep, q + 1, p)),
        X_post_mean=np.zeros((N, p)),
        sel=sel,
    )
    t0 = time.time()
    for it in range(n_iter):
        # (1) outcome: Polya-gamma + Gaussian update of (a, b_z, b_x)
        D = np.column_stack([np.ones(N), Z, Xc])
        eta = D @ beta
        om = sample_pg1(eta, rng)
        beta = _chol_sample((D * om[:, None]).T @ D + P0, D.T @ kappa, rng)
        eta = D @ beta

        # (2) selection models: latent probit scores, then coefficients
        for j in sel:
            g = gam[j]
            w[:, j] = draw_latent_probit(M @ g[:km] + g[km] * Xc[:, j], obs[:, j], rng)
            if fixed[j] is None:
                G, target = np.column_stack([M, Xc[:, j]]), w[:, j]
                prec0 = np.diag(1 / prior_sd[j] ** 2)
                m0 = prior_m[j]
            else:
                G, target = M, w[:, j] - fixed[j] * Xc[:, j]
                prec0 = np.diag(1 / prior_sd[j][:km] ** 2)
                m0 = prior_m[j][:km]
            draw = _chol_sample(G.T @ G + prec0, G.T @ target + prec0 @ m0, rng)
            gam[j] = np.append(draw, g[km]) if fixed[j] is not None else draw

        # (3) missing vitals: full conditionals (Gaussian)
        mu = Zc @ B
        Om = np.linalg.inv(Sigma)
        R = Xc - mu
        for j in range(p):
            idx = miss[:, j]
            if not idx.any():
                continue
            Rj = R[idx]
            cond_mean = mu[idx, j] - (Rj @ Om[:, j] - Rj[:, j] * Om[j, j]) / Om[j, j]
            prec = np.full(idx.sum(), Om[j, j])
            lin = Om[j, j] * cond_mean
            bj = beta[1 + q + j]
            eta_minus = eta[idx] - bj * Xc[idx, j]
            prec = prec + om[idx] * bj ** 2
            lin = lin + bj * (kappa[idx] - om[idx] * eta_minus)
            if j in gam:
                gx = gam[j][km]
                prec = prec + gx ** 2
                lin = lin + gx * (w[idx, j] - M[idx] @ gam[j][:km])
            var = 1.0 / prec
            xnew = lin * var + np.sqrt(var) * rng.standard_normal(idx.sum())
            eta[idx] += bj * (xnew - Xc[idx, j])
            Xc[idx, j] = xnew
            R[idx, j] = xnew - mu[idx, j]

        # (4) vitals regression and covariance
        Mn = V_B @ Zc.T @ Xc
        E = Xc - Zc @ Mn
        S = Psi0 + E.T @ E + Mn.T @ K0 @ Mn
        Sigma = np.atleast_2d(invwishart.rvs(df=nu0 + N, scale=S, random_state=rng))
        B = Mn + L_VB @ rng.standard_normal((q + 1, p)) @ np.linalg.cholesky(Sigma).T

        if it >= n_burn:
            k = it - n_burn
            out["beta"][k] = beta
            for a, j in enumerate(sel):
                out["gamma"][k, a] = gam[j]
            out["imp_mean"][k] = [Xc[miss[:, j], j].mean() if miss[:, j].any() else np.nan for j in range(p)]
            out["Sigma"][k] = Sigma
            out["B"][k] = B
            out["X_post_mean"] += Xc / n_keep
        if verbose and (it + 1) % 500 == 0:
            print(f"  [seed {seed}] iter {it + 1}/{n_iter}  {time.time() - t0:.0f}s", flush=True)
    return out


def _run(args):
    return run_chain(**args)


def fit(data, selection=None, chains=2, n_iter=3000, n_burn=1000, seed=1, verbose=True, **kw):
    """Run several chains in parallel. Returns dict with arrays shaped (chains, draws, ...)."""
    selection = selection or {}
    jobs = [dict(Z=data["Z"], X=data["X"], y=data["y"], selection=selection, names=data["vital_cols"],
                 n_iter=n_iter, n_burn=n_burn, seed=seed * 1000 + c, verbose=verbose and c == 0, **kw)
            for c in range(chains)]
    if chains == 1:
        res = [_run(jobs[0])]
    else:
        with ProcessPoolExecutor(max_workers=min(chains, os.cpu_count() or 1)) as ex:
            res = list(ex.map(_run, jobs))
    stack = lambda k: np.stack([r[k] for r in res])
    return dict(beta=stack("beta"), gamma=stack("gamma"), imp_mean=stack("imp_mean"),
                Sigma=stack("Sigma"), B=stack("B"), X_post_mean=np.mean([r["X_post_mean"] for r in res], 0),
                sel=res[0]["sel"], data=data, selection=selection)


# --------------------------------------------------------------------------- diagnostics
def rhat(x):
    """Split R-hat. x: (chains, n)."""
    m, n = x.shape
    h = n // 2
    s = np.concatenate([x[:, :h], x[:, h:2 * h]], 0)
    n_, m_ = s.shape[1], s.shape[0]
    W = s.var(1, ddof=1).mean()
    Bn = s.mean(1).var(ddof=1)
    return float(np.sqrt(((n_ - 1) / n_ * W + Bn) / W)) if W > 0 else np.nan


def ess(x):
    """Effective sample size (Geyer initial positive sequence). x: (chains, n)."""
    m, n = x.shape
    xc = x - x.mean(1, keepdims=True)
    f = np.fft.rfft(xc, 2 * n, axis=1)
    acov = np.fft.irfft(f * np.conj(f), axis=1)[:, :n] / n * (n / (n - 1))
    W = acov[:, 0].mean()
    var_plus = (n - 1) / n * W + (x.mean(1).var(ddof=1) if m > 1 else 0.0)
    if var_plus <= 0:
        return np.nan
    rho = 1 - (W - acov.mean(0)) / var_plus
    tau, t = 0.0, 0
    while t + 1 < n:
        pair = rho[t] + rho[t + 1]
        if pair < 0:
            break
        tau += pair
        t += 2
    tau = max(-1 + 2 * tau, 1e-3)
    return float(m * n / tau)


def summarize(draws, labels, scale=None):
    """draws: (chains, n, k). Optional `scale` (k,) multiplies draws (unit conversion)."""
    d = draws if scale is None else draws * np.asarray(scale)
    flat = d.reshape(-1, d.shape[-1])
    return pd.DataFrame({
        "mean": flat.mean(0), "sd": flat.std(0),
        "q2.5": np.quantile(flat, 0.025, 0), "q97.5": np.quantile(flat, 0.975, 0),
        "rhat": [rhat(draws[:, :, k]) for k in range(d.shape[-1])],
        "ess": [ess(draws[:, :, k]) for k in range(d.shape[-1])],
    }, index=labels)


def coef_labels(res):
    return ["intercept"] + res["data"]["base_cols"] + res["data"]["vital_cols"]


def coef_scale(res):
    """Multiplier converting model-scale slopes to 'per original SD' units (1 for base vars)."""
    q = len(res["data"]["base_cols"])
    return np.concatenate([np.ones(1 + q), res["data"]["x_scale"]])


In [4]:
#!/usr/bin/env python3
"""
run_analysis.py -- fit the joint Bayesian missing-data model to the synthetic EHR data,
compare it with standard approaches, check convergence, and run a sensitivity analysis.

Methods compared (all coefficients reported per 1 SD of the TRUE vital / base variable,
i.e. the scale the truth was generated on):

  oracle        logistic regression on the complete data (before deletion; unattainable)
  complete-case logistic regression on patients with all 11 vitals observed
  mean-imput.   mean imputation + logistic regression
  MICE          chained equations (20 imputations, Rubin's rules), outcome in imputation model
  Bayes-MAR     joint Bayesian model, missingness ignorable (MAR)
  Bayes-MNAR    joint Bayesian model + selection model for every vital (g_x free)

Then a delta-shift sensitivity analysis for the suspected-MNAR vitals.
"""
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import argparse
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit

import bayes_missing as bm

warnings.filterwarnings("ignore")

LOG_VARS = ("glucose", "lactate", "creatinine", "wbc")
# Vitals a clinician would flag as plausibly MNAR, with the assumed DIRECTION of the effect:
# +1 = higher values more likely to be recorded, -1 = lower values more likely to be recorded.
SUSPECT = {"glucose": +1, "lactate": +1, "creatinine": +1, "sodium": -1}
SENS_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]      # |g_x| per SD, probit scale


# ------------------------------------------------------------------ frequentist helpers
def logit_fit(D, y, iters=100):
    b = np.zeros(D.shape[1])
    for _ in range(iters):
        pr = expit(D @ b)
        H = (D * (pr * (1 - pr))[:, None]).T @ D + 1e-8 * np.eye(D.shape[1])
        step = np.linalg.solve(H, D.T @ (y - pr))
        b += step
        if np.abs(step).max() < 1e-8:
            break
    return b, np.linalg.inv(H)


def freq_table(b, cov, to_true):
    est, se = b[1:] * to_true, np.sqrt(np.diag(cov))[1:] * to_true
    return pd.DataFrame({"est": est, "lo": est - 1.96 * se, "hi": est + 1.96 * se})


def design(Z, X):
    return np.column_stack([np.ones(len(Z)), Z, X])


def mice_fit(Z, X, y, to_true, m=20):
    from sklearn.experimental import enable_iterative_imputer  # noqa: F401
    from sklearn.impute import IterativeImputer
    from sklearn.linear_model import BayesianRidge
    A = np.column_stack([Z, X, y])
    bs, vs = [], []
    for k in range(m):
        imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, max_iter=10, random_state=k)
        Ai = imp.fit_transform(A)
        b, cov = logit_fit(design(Ai[:, :Z.shape[1]], Ai[:, Z.shape[1]:-1]), y)
        bs.append(b)
        vs.append(np.diag(cov))
    bs, vs = np.array(bs), np.array(vs)
    qbar = bs.mean(0)
    T = vs.mean(0) + (1 + 1 / m) * bs.var(0, ddof=1)          # Rubin's rules
    est, se = qbar[1:] * to_true, np.sqrt(T)[1:] * to_true
    return pd.DataFrame({"est": est, "lo": est - 1.96 * se, "hi": est + 1.96 * se})


def bayes_table(res, to_true):
    labels = bm.coef_labels(res)
    s = bm.summarize(res["beta"], labels, scale=np.concatenate([[1.0], to_true]))
    s = s.iloc[1:]
    return s, pd.DataFrame({"est": s["mean"].values, "lo": s["q2.5"].values, "hi": s["q97.5"].values})


def metrics(tab, truth, mnar_idx):
    err = tab["est"].values - truth
    return dict(RMSE_all=float(np.sqrt((err ** 2).mean())),
                RMSE_mnar_vitals=float(np.sqrt((err[mnar_idx] ** 2).mean())),
                coverage=float(((tab["lo"].values <= truth) & (truth <= tab["hi"].values)).mean()),
                mean_width=float((tab["hi"] - tab["lo"]).mean()))


# ------------------------------------------------------------------ plotting
INK, INK2, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"


def _style(ax, title):
    ax.set_facecolor(SURF)
    ax.set_title(title, loc="left", fontsize=11, color=INK, fontweight="bold")
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def make_recovery_figure(coef, imp, gam, labels, vit, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 7.2), facecolor=SURF,
                             gridspec_kw=dict(width_ratios=[1.25, 1, 1]))
    # A: estimation error
    ax = axes[0]
    _style(ax, "A  Error in outcome coefficients (estimate - truth)")
    meth = [("Complete-case", C_BLUE), ("MICE", C_ORANGE), ("Bayes, MAR", C_AQUA), ("Bayes, MNAR selection", C_YELLOW)]
    off = np.linspace(-0.27, 0.27, 4)
    truth = coef["truth"].values
    for (name, col), o in zip(meth, off):
        t = coef[name]
        y = np.arange(len(labels)) + o
        ax.hlines(y, t["lo"] - truth, t["hi"] - truth, color=col, lw=1.6, alpha=0.9)
        ax.plot(t["est"] - truth, y, "o", color=col, ms=6, mec=SURF, mew=1.5, label=name)
    ax.axvline(0, color=INK, lw=1.2)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, color=INK2)
    ax.invert_yaxis()
    ax.set_xlabel("log-odds per 1 SD (0 = truth)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    # B: imputation check
    ax = axes[1]
    _style(ax, "B  Mean of the missing values: imputed - true")
    y = np.arange(len(vit))
    ax.plot(imp["observed_mean"] - imp["true_missing_mean"], y, "o", mfc=SURF, mec=MUTED, mew=1.6, ms=7,
            label="Observed mean (no adjustment)")
    ax.plot(imp["bayes_mar"] - imp["true_missing_mean"], y, "o", color=C_AQUA, ms=7, mec=SURF, mew=1.5, label="Bayes, MAR")
    ax.plot(imp["bayes_mnar"] - imp["true_missing_mean"], y, "o", color=C_YELLOW, ms=7, mec=SURF, mew=1.5,
            label="Bayes, MNAR selection")
    ax.axvline(0, color=INK, lw=1.2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{v}  ({m})" for v, m in zip(vit, imp["mech"])], color=INK2, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlabel("difference, in SDs of the observed values (0 = unbiased)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    # C: g_x
    ax = axes[2]
    _style(ax, "C  Own-value effect on being observed (g_x)")
    ax.hlines(y, gam["lo"], gam["hi"], color=C_YELLOW, lw=1.8)
    ax.plot(gam["est"], y, "o", color=C_YELLOW, ms=7, mec=SURF, mew=1.5, label="Posterior mean, 95% CrI")
    ax.plot(gam["true"], y, "D", color=INK, ms=6, mec=SURF, mew=1.2, label="Truth")
    ax.axvline(0, color=MUTED, lw=1.0, ls=":")
    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.set_xlabel("probit coefficient per 1 SD of the vital (0 = ignorable)", color=INK2, fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, labelcolor=INK2)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor=SURF)
    plt.close(fig)


def make_sensitivity_figure(sens, truth_by_vital, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8.5, 5.2), facecolor=SURF)
    _style(ax, "Sensitivity of outcome coefficients to the assumed MNAR strength")
    ax.grid(axis="y", color=GRID, lw=0.8)
    cols = dict(zip(SUSPECT, [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW]))
    for v, c in cols.items():
        est, lo, hi = sens[f"{v}__est"], sens[f"{v}__lo"], sens[f"{v}__hi"]
        ax.fill_between(sens["s"], lo, hi, color=c, alpha=0.12, lw=0)
        ax.plot(sens["s"], est, "-o", color=c, lw=2, ms=6, mec=SURF, mew=1.5, label=v)
        ax.axhline(truth_by_vital[v], color=c, lw=1, ls="--")
        ax.text(sens["s"].iloc[-1] + 0.02, est.iloc[-1], v, color=INK2, fontsize=9, va="center")
    ax.set_xlabel("assumed |g_x| (probit, per SD) -- higher values more likely recorded (sodium: lower)", color=INK2, fontsize=9)
    ax.set_ylabel("log-odds per 1 SD (dashed = truth)", color=INK2, fontsize=9)
    ax.set_xlim(-0.03, 1.2)
    ax.legend(frameon=False, fontsize=9, loc="upper right", labelcolor=INK2, ncol=2)
    fig.tight_layout()
    fig.savefig(path, dpi=140, facecolor=SURF)
    plt.close(fig)


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    here = Path.cwd()
    if not (here / 'bayes_missing.py').exists() and (here / 'Preprocessing' / 'bayes_missing.py').exists():
        here = here / 'Preprocessing'
    ap.add_argument("--data-dir", default=str(here / "data"))
    ap.add_argument("--out-dir", default=str(here / "results"))
    ap.add_argument("--iter", type=int, default=6000)
    ap.add_argument("--burn", type=int, default=2500)
    ap.add_argument("--chains", type=int, default=2)
    ap.add_argument("--skip-sensitivity", action="store_true")
    a = ap.parse_args([])  # notebook: use defaults (parsing the kernel's argv would fail)
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(Path(a.data_dir) / "synthetic_ehr.csv")
    full = pd.read_csv(Path(a.data_dir) / "synthetic_ehr_complete.csv")
    truth = json.loads((Path(a.data_dir) / "truth.json").read_text())
    base, vit = truth["base"], truth["vitals"]
    q, p = len(base), len(vit)
    spread = np.array(truth["vital_spread"])
    beta_true = np.array(truth["beta_base"] + truth["beta_vitals"])
    labels = base + vit
    mnar_idx = np.array([q + vit.index(v) for v in ("glucose", "lactate", "creatinine", "sodium")])

    data = bm.prepare(df, base, vit, "outcome", LOG_VARS)
    factor = data["x_scale"] / spread                      # model slope -> per true SD
    to_true = np.concatenate([np.ones(q), factor])
    Z, X, y = data["Z"], data["X"], data["y"]
    coef = {"truth": pd.Series(beta_true)}

    print("== baselines", flush=True)
    dfull = bm.prepare(full, base, vit, "outcome", LOG_VARS)
    b, cov = logit_fit(design(dfull["Z"], dfull["X"]), dfull["y"])
    coef["Oracle (complete data)"] = freq_table(b, cov, np.concatenate([np.ones(q), dfull["x_scale"] / spread]))
    cc = ~np.isnan(X).any(1)
    b, cov = logit_fit(design(Z[cc], X[cc]), y[cc])
    coef["Complete-case"] = freq_table(b, cov, to_true)
    print(f"   complete cases: {cc.sum()} of {len(cc)}", flush=True)
    b, cov = logit_fit(design(Z, np.nan_to_num(X)), y)
    coef["Mean imputation"] = freq_table(b, cov, to_true)
    coef["MICE"] = mice_fit(Z, X, y, to_true)

    print("== Bayesian model, MAR (ignorable)", flush=True)
    t0 = time.time()
    res_mar = bm.fit(data, {}, chains=a.chains, n_iter=a.iter, n_burn=a.burn, seed=1)
    diag_mar, coef["Bayes, MAR"] = bayes_table(res_mar, to_true)
    print(f"   {time.time() - t0:.0f}s", flush=True)

    print("== Bayesian model, selection model for every vital (g_x free)", flush=True)
    t0 = time.time()
    res_sel = bm.fit(data, {v: None for v in vit}, chains=a.chains, n_iter=a.iter, n_burn=a.burn, seed=2)
    diag_sel, coef["Bayes, MNAR selection"] = bayes_table(res_sel, to_true)
    print(f"   {time.time() - t0:.0f}s", flush=True)

    # ---- coefficient table
    wide = pd.DataFrame({"truth": beta_true}, index=labels)
    for k, v in coef.items():
        if k == "truth":
            continue
        v = v.copy()
        v.index = labels
        wide[k] = v["est"]
        wide[k + " lo"] = v["lo"]
        wide[k + " hi"] = v["hi"]
    wide.round(3).to_csv(out / "coefficients.csv")
    est_cols = ["truth", "Oracle (complete data)", "Complete-case", "Mean imputation", "MICE",
                "Bayes, MAR", "Bayes, MNAR selection"]
    print("\nOutcome coefficients (log-odds per 1 SD)")
    print(wide[est_cols].round(2).to_string())

    rows = {}
    for k in est_cols[1:]:
        rows[k] = metrics(coef[k], beta_true, mnar_idx)
    mt = pd.DataFrame(rows).T
    mt.round(3).to_csv(out / "metrics.csv")
    print("\nAccuracy against the truth (single simulated dataset)")
    print(mt.round(3).to_string())

    # ---- convergence
    dg = pd.concat([diag_mar.add_prefix("MAR_"), diag_sel.add_prefix("MNAR_")], axis=1)
    dg.round(3).to_csv(out / "convergence_beta.csv")
    print(f"\nConvergence (outcome coefficients): max R-hat  MAR={diag_mar['rhat'].max():.3f}  "
          f"MNAR={diag_sel['rhat'].max():.3f};  min ESS  MAR={diag_mar['ess'].min():.0f}  MNAR={diag_sel['ess'].min():.0f}")

    # ---- selection-model coefficients
    sel = res_sel["sel"]
    km = q + 2
    g = res_sel["gamma"]                                    # (chains, n, nsel, km+1)
    rows = []
    for ai, j in enumerate(sel):
        name = vit[j]
        m = truth["missingness"][name]
        dx = g[:, :, ai, km] / factor[j]                    # per true SD
        dy = g[:, :, ai, q + 1]
        flat = dx.reshape(-1)
        rows.append(dict(vital=name, mechanism=m["mechanism"], true_gx=m["gx"], gx_mean=flat.mean(),
                         gx_lo=np.quantile(flat, .025), gx_hi=np.quantile(flat, .975),
                         gx_rhat=bm.rhat(dx), true_gy=m["gy"], gy_mean=dy.mean(),
                         gy_lo=np.quantile(dy, .025), gy_hi=np.quantile(dy, .975)))
    gtab = pd.DataFrame(rows)
    gtab.round(3).to_csv(out / "missingness_model.csv", index=False)
    print("\nSelection-model coefficients (probit, per SD of the vital; g_y = effect of the outcome)")
    print(gtab.round(2).to_string(index=False))

    # ---- imputation check
    full_t = full[vit].to_numpy(float).copy()
    for j, nm in enumerate(vit):
        if nm in LOG_VARS:
            full_t[:, j] = np.log(full_t[:, j])
    full_m = (full_t - data["x_loc"]) / data["x_scale"]
    miss = np.isnan(X)
    true_miss_mean = np.array([full_m[miss[:, j], j].mean() for j in range(p)])
    imp = pd.DataFrame({
        "vital": vit, "mech": [truth["missingness"][v]["mechanism"].split(" (")[0].split(" +")[0] for v in vit],
        "observed_mean": 0.0, "true_missing_mean": true_miss_mean,
        "bayes_mar": res_mar["imp_mean"].mean((0, 1)), "bayes_mnar": res_sel["imp_mean"].mean((0, 1)),
    })
    imp.round(3).to_csv(out / "imputation_check.csv", index=False)
    print("\nMean of missing values (observed-data SD units)")
    print(imp.round(2).to_string(index=False))

    make_recovery_figure(
        dict(truth=pd.Series(beta_true), **{k: coef[k] for k in ("Complete-case", "MICE", "Bayes, MAR", "Bayes, MNAR selection")}),
        imp, dict(est=gtab["gx_mean"].values, lo=gtab["gx_lo"].values, hi=gtab["gx_hi"].values, true=gtab["true_gx"].values),
        labels, vit, out / "fig_recovery.png")
    np.savez_compressed(out / "posterior_draws.npz", beta_mar=res_mar["beta"], beta_mnar=res_sel["beta"],
                        gamma_mnar=res_sel["gamma"], labels=np.array(bm.coef_labels(res_sel)))

    # ---- sensitivity analysis
    if not a.skip_sensitivity:
        print("\n== sensitivity analysis (g_x fixed for suspect vitals)", flush=True)
        recs = []
        for s in SENS_GRID:
            t0 = time.time()
            selection = {v: d * s * factor[vit.index(v)] for v, d in SUSPECT.items()}
            r = bm.fit(data, selection, chains=a.chains, n_iter=3000, n_burn=1000, seed=10 + int(s * 100), verbose=False)
            _, tab = bayes_table(r, to_true)
            rec = {"s": s}
            for v in vit:
                rec[f"{v}__est"], rec[f"{v}__lo"], rec[f"{v}__hi"] = tab.loc[labels.index(v), ["est", "lo", "hi"]]
            recs.append(rec)
            print(f"   s={s:.2f}  {time.time() - t0:.0f}s", flush=True)
        sens = pd.DataFrame(recs)
        sens.round(3).to_csv(out / "sensitivity.csv", index=False)
        for v in SUSPECT:
            print(f"   {v:11s} beta by s:", ", ".join(f"{x:.2f}" for x in sens[f"{v}__est"]),
                  f"  (truth {beta_true[labels.index(v)]:.2f})")
        make_sensitivity_figure(sens, {v: beta_true[labels.index(v)] for v in SUSPECT}, out / "fig_sensitivity.png")
        make_recovery_figure(
            dict(truth=pd.Series(beta_true), **{k: coef[k] for k in ("Complete-case", "MICE", "Bayes, MAR", "Bayes, MNAR selection")}),
            imp, dict(est=gtab["gx_mean"].values, lo=gtab["gx_lo"].values, hi=gtab["gx_hi"].values, true=gtab["true_gx"].values),
            labels, vit, out / "fig_recovery.png") 
    print("\ndone ->", out)


if __name__ == "__main__":
    main()


== baselines
   complete cases: 532 of 3000
== Bayesian model, MAR (ignorable)
  [seed 1000] iter 500/6000  4s
  [seed 1000] iter 1000/6000  8s
  [seed 1000] iter 1500/6000  13s
  [seed 1000] iter 2000/6000  17s
  [seed 1000] iter 2500/6000  21s
  [seed 1000] iter 3000/6000  26s
  [seed 1000] iter 3500/6000  30s
  [seed 1000] iter 4000/6000  34s
  [seed 1000] iter 4500/6000  39s
  [seed 1000] iter 5000/6000  43s
  [seed 1000] iter 5500/6000  48s
  [seed 1000] iter 6000/6000  52s
   53s
== Bayesian model, selection model for every vital (g_x free)
  [seed 2000] iter 500/6000  7s
  [seed 2000] iter 1000/6000  14s
  [seed 2000] iter 1500/6000  21s
  [seed 2000] iter 2000/6000  28s
  [seed 2000] iter 2500/6000  35s
  [seed 2000] iter 3000/6000  42s
  [seed 2000] iter 3500/6000  49s
  [seed 2000] iter 4000/6000  56s
  [seed 2000] iter 4500/6000  64s
  [seed 2000] iter 5000/6000  72s
  [seed 2000] iter 5500/6000  80s
  [seed 2000] iter 6000/6000  87s
   88s

Outcome coefficients (log-odds pe